# DISCERN & mini-DISCERN — Synthetic Report Test

Smoke-tests both the full **DISCERN** pipeline (3-stage: entity extraction → attribute comparison → significance scoring) and the single-prompt **mini-DISCERN** evaluator on a handful of synthetic chest X-ray report pairs.

Backbone model: `databricks-claude-sonnet-4-6` (credentials read from `config.yaml`).

## 1. Environment setup

Adds `discern/src` to the import path and loads Databricks credentials from `config.yaml`.

In [1]:
import os, sys
from pathlib import Path
import yaml

REPO_ROOT = Path('/vast/projects/witschey/pmbb-vision/research_projects/rakshrma/workspace/discern')
sys.path.insert(0, str(REPO_ROOT / 'src'))

cfg   = yaml.safe_load((REPO_ROOT / 'config.yaml').read_text())
creds = cfg.get('credentials', {}) or {}
db_token = creds.get('databricks_token', '')
db_host  = creds.get('databricks_host',  '')
if db_host:
    os.environ['DATABRICKS_SERVING_ENDPOINTS_URL'] = db_host

MODEL = 'databricks-claude-sonnet-4-6'

PROMPT_YAML       = str(REPO_ROOT / 'config/entity_extraction_prompt.yaml')
ENTITIES_YAML     = str(REPO_ROOT / 'config/entities.yaml')
ATTRIBUTE_PROMPT  = str(REPO_ROOT / 'config/attribute_extraction_prompt.yaml')
SIGNIFICANCE_YAML = str(REPO_ROOT / 'config/significance_prompt.yaml')
DIAG_ENTITIES     = str(REPO_ROOT / 'config/diagnosis.yaml')
MERGED_PROMPT     = str(REPO_ROOT / 'config/merged_prompt.yaml')

assert db_token, 'No databricks_token in config.yaml'
assert db_host,  'No databricks_host in config.yaml'
print(f'Model    : {MODEL}')
print(f'Endpoint : {db_host}')

Model    : databricks-claude-sonnet-4-6
Endpoint : https://adb-624977420987022.2.azuredatabricks.net/serving-endpoints


## 2. Synthetic report pairs

Five scenarios chosen to exercise different parts of the scorer:

| # | Scenario | Expected behavior |
|---|----------|------------------|
| 1 | Concordant — same findings, slightly reworded | low/zero score |
| 2 | Severity mismatch — `moderate` vs `small` effusion | low–moderate score |
| 3 | Missed acute finding — pneumonia missed by candidate | high score |
| 4 | Hallucinated finding — candidate adds a suspicious nodule | high score |
| 5 | Multiple discrepancies — pneumothorax + cardiomegaly mismatch | high score |

In [2]:
EXAMPLES = [
    {
        'name': '1_concordant',
        'reference': (
            'PA and lateral chest radiograph. The lungs are clear. '
            'No focal consolidation, pleural effusion, or pneumothorax. '
            'Cardiac silhouette is normal in size. '
            'IMPRESSION: No acute cardiopulmonary abnormality.'
        ),
        'candidate': (
            'PA and lateral chest radiograph. The lungs are clear bilaterally. '
            'No focal consolidation, effusion, or pneumothorax. '
            'The heart is normal in size. '
            'IMPRESSION: No acute cardiopulmonary findings.'
        ),
    },
    {
        'name': '2_severity_mismatch',
        'reference': (
            'PA and lateral chest radiograph. There is a moderate left pleural effusion '
            'with associated left lower lobe atelectasis. The right lung is clear. '
            'The cardiac silhouette is mildly enlarged. No pneumothorax. '
            'IMPRESSION: Moderate left pleural effusion with left lower lobe atelectasis. '
            'Mild cardiomegaly.'
        ),
        'candidate': (
            'PA and lateral chest radiograph. There is a small left pleural effusion. '
            'No consolidation or pneumothorax identified. '
            'The cardiac silhouette is normal in size. '
            'The mediastinum is unremarkable. '
            'IMPRESSION: Small left pleural effusion.'
        ),
    },
    {
        'name': '3_missed_pneumonia',
        'reference': (
            'Frontal and lateral chest radiograph. There is dense consolidation in the '
            'right lower lobe consistent with pneumonia. No pleural effusion or pneumothorax. '
            'The heart size is normal. '
            'IMPRESSION: Right lower lobe pneumonia.'
        ),
        'candidate': (
            'Frontal and lateral chest radiograph. The lungs are clear bilaterally. '
            'No pleural effusion or pneumothorax. '
            'Cardiac and mediastinal contours are unremarkable. '
            'IMPRESSION: Normal chest radiograph.'
        ),
    },
    {
        'name': '4_hallucinated_nodule',
        'reference': (
            'Frontal chest radiograph. The lungs are clear. '
            'No pleural effusion or pneumothorax. Cardiac silhouette is normal. '
            'IMPRESSION: Normal chest radiograph.'
        ),
        'candidate': (
            'Frontal chest radiograph. There is a 2 cm right upper lobe nodule which is '
            'suspicious for malignancy. No pleural effusion. Cardiac silhouette is normal. '
            'IMPRESSION: Right upper lobe nodule, recommend CT for further characterization.'
        ),
    },
    {
        'name': '5_multiple_discrepancies',
        'reference': (
            'Portable AP chest radiograph. There is a small right apical pneumothorax. '
            'The cardiac silhouette is enlarged. Bilateral patchy opacities are present, '
            'concerning for pulmonary edema. No focal consolidation. '
            'IMPRESSION: 1) Small right apical pneumothorax. '
            '2) Cardiomegaly with pulmonary edema.'
        ),
        'candidate': (
            'Portable AP chest radiograph. The lungs are clear without focal opacity. '
            'No pneumothorax. Cardiac silhouette is normal in size. '
            'IMPRESSION: No acute cardiopulmonary abnormality.'
        ),
    },
]

for ex in EXAMPLES:
    print(f"{ex['name']:<28s}  ref={len(ex['reference'])} chars  cand={len(ex['candidate'])} chars")

1_concordant                  ref=200 chars  cand=192 chars
2_severity_mismatch           ref=300 chars  cand=233 chars
3_missed_pneumonia            ref=220 chars  cand=195 chars
4_hallucinated_nodule         ref=150 chars  cand=232 chars
5_multiple_discrepancies      ref=291 chars  cand=177 chars


## 3. Full DISCERN pipeline

Three sequential LLM calls per example: entity extraction (×2 for ref+cand) → attribute comparison → significance scoring. Returns a per-entity evaluation list and an aggregate `discern_score`.

In [3]:
from evaluate_reports import run_evaluation

discern_results = []
for ex in EXAMPLES:
    print(f"\n{'='*70}\n[DISCERN] {ex['name']}\n{'='*70}")
    try:
        evaluation, score = run_evaluation(
            report_text         = ex['reference'],
            candidate_text      = ex['candidate'],
            model               = MODEL,
            token_path          = db_token,
            prompt_yaml_path    = PROMPT_YAML,
            entities_yaml_path  = ENTITIES_YAML,
            attribute_prompt_path  = ATTRIBUTE_PROMPT,
            significance_yaml_path = SIGNIFICANCE_YAML,
            max_tokens          = 5000,
        )
        discern_results.append({'name': ex['name'], 'score': score, 'evaluation': evaluation, 'error': None})
        print(f"  ✓ score = {score}   |   {len(evaluation)} entities scored")
    except Exception as exc:
        discern_results.append({'name': ex['name'], 'score': None, 'evaluation': None, 'error': str(exc)})
        print(f"  ✗ FAILED: {exc}")


[DISCERN] 1_concordant
[DB] prompt_tokens=2546  completion_tokens=240  total=2786  time=5.63s  tok/s=42.6
{'prompt_tokens': 2546, 'completion_tokens': 240, 'total_tokens': 2786}
[DB] prompt_tokens=2542  completion_tokens=225  total=2767  time=3.12s  tok/s=72.2
{'prompt_tokens': 2542, 'completion_tokens': 225, 'total_tokens': 2767}
Entity extraction time taken: 9.04s
[DB] prompt_tokens=1437  completion_tokens=487  total=1924  time=4.65s  tok/s=104.8
Attribute comparison time taken: 4.65s
Entity merge time taken: 0.00s
[DB] prompt_tokens=1272  completion_tokens=290  total=1562  time=4.32s  tok/s=67.2
{'prompt_tokens': 1272, 'completion_tokens': 290, 'total_tokens': 1562}
Significance evaluation time taken: 4.33s
Final merge time taken: 0.00s
Score calculation time taken: 0.00s
Total pipeline time: 18.02s
  ✓ score = 0.0   |   4 entities scored

[DISCERN] 2_severity_mismatch
[DB] prompt_tokens=2569  completion_tokens=283  total=2852  time=4.27s  tok/s=66.3
{'prompt_tokens': 2569, 'comple

## 4. mini-DISCERN (single-prompt) evaluator

One LLM call per pair. Returns a list of `EntityEvaluation` Pydantic objects; the aggregate score is the sum of `clinical_significance_score` across entities.

In [4]:
from evaluate_single_prompt import evaluate_reports

mini_results = []
for ex in EXAMPLES:
    print(f"\n{'='*70}\n[mini-DISCERN] {ex['name']}\n{'='*70}")
    try:
        entities = evaluate_reports(
            reference_report  = ex['reference'],
            candidate_report  = ex['candidate'],
            entity_list_path  = DIAG_ENTITIES,
            prompt_path       = MERGED_PROMPT,
            model             = MODEL,
            token_path        = db_token,
            max_tokens        = 8000,
            temperature       = 0.1,
            max_retries       = 3,
        )
        score = int(sum(e.clinical_significance_score for e in entities))
        mini_results.append({'name': ex['name'], 'score': score, 'entities': entities, 'error': None})
        print(f"  ✓ score = {score}   |   {len(entities)} entities scored")
    except Exception as exc:
        mini_results.append({'name': ex['name'], 'score': None, 'entities': None, 'error': str(exc)})
        print(f"  ✗ FAILED: {exc}")


[mini-DISCERN] 1_concordant
[DB] prompt_tokens=2721  completion_tokens=4  total=2725  time=1.33s  tok/s=3.0
  ✓ score = 0   |   0 entities scored

[mini-DISCERN] 2_severity_mismatch
[DB] prompt_tokens=2751  completion_tokens=595  total=3346  time=6.47s  tok/s=91.9
  ✓ score = 6   |   3 entities scored

[mini-DISCERN] 3_missed_pneumonia
[DB] prompt_tokens=2721  completion_tokens=212  total=2933  time=3.36s  tok/s=63.0
  ✓ score = 4   |   1 entities scored

[mini-DISCERN] 4_hallucinated_nodule
[DB] prompt_tokens=2718  completion_tokens=221  total=2939  time=3.68s  tok/s=60.1
  ✓ score = 4   |   1 entities scored

[mini-DISCERN] 5_multiple_discrepancies
[DB] prompt_tokens=2746  completion_tokens=590  total=3336  time=6.70s  tok/s=88.1
  ✓ score = 12   |   3 entities scored


## 5. Side-by-side comparison

In [6]:
import pandas as pd

rows = []
for d, m in zip(discern_results, mini_results):
    rows.append({
        'example'            : d['name'],
        'DISCERN_score'      : d['score'],
        'DISCERN_n_entities' : len(d['evaluation']) if d['evaluation'] is not None else None,
        'miniDISCERN_score'  : m['score'],
        'miniDISCERN_n_entities': len(m['entities']) if m['entities'] is not None else None,
    })
summary = pd.DataFrame(rows)
summary

,example,DISCERN_score,DISCERN_n_entities,miniDISCERN_score,miniDISCERN_n_entities
0,1_concordant,0.0,4,0,0
1,2_severity_mismatch,10.0,6,6,3
2,3_missed_pneumonia,9.0,5,4,1
3,4_hallucinated_nodule,10.0,4,4,1
4,5_multiple_discrepancies,19.0,5,12,3


## 6. Drill into a single example

Inspect the per-entity output of both pipelines for one of the discordant examples. Change `idx` to look at others.

In [7]:
import json

idx = 2  # 0=concordant, 1=severity, 2=missed pneumonia, 3=hallucinated, 4=multiple

print(f"--- Example: {EXAMPLES[idx]['name']} ---\n")
print('REFERENCE:')
print(EXAMPLES[idx]['reference'])
print('\nCANDIDATE:')
print(EXAMPLES[idx]['candidate'])

print('\n\n=== Full DISCERN per-entity output ===')
if discern_results[idx]['evaluation']:
    for entity in discern_results[idx]['evaluation']:
        print(json.dumps(entity, indent=2, default=str))
else:
    print(f"(failed: {discern_results[idx]['error']})")

print('\n\n=== mini-DISCERN per-entity output ===')
if mini_results[idx]['entities']:
    for e in mini_results[idx]['entities']:
        print(json.dumps(e.model_dump(), indent=2, default=str))
else:
    print(f"(failed: {mini_results[idx]['error']})")

--- Example: 3_missed_pneumonia ---

REFERENCE:
Frontal and lateral chest radiograph. There is dense consolidation in the right lower lobe consistent with pneumonia. No pleural effusion or pneumothorax. The heart size is normal. IMPRESSION: Right lower lobe pneumonia.

CANDIDATE:
Frontal and lateral chest radiograph. The lungs are clear bilaterally. No pleural effusion or pneumothorax. Cardiac and mediastinal contours are unremarkable. IMPRESSION: Normal chest radiograph.


=== Full DISCERN per-entity output ===
{
  "entity": "Lung and Pleural Opacity :: Pleural Effusion",
  "reference_report_finding": "No pleural effusion or pneumothorax.",
  "candidate_report_finding": "No pleural effusion or pneumothorax.",
  "diagnosis_concordance": "concordant",
  "location_concordance": "not mentioned",
  "severity_concordance": "not mentioned",
  "temporal_comparison": "not mentioned",
  "significance_score": 0,
  "rationale": "Both reports concordantly note no pleural effusion, representing ful

## 7. (Optional) Save results to JSON

In [ ]:
out_dir = REPO_ROOT / 'notebooks' / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

payload = []
for ex, d, m in zip(EXAMPLES, discern_results, mini_results):
    payload.append({
        'name'           : ex['name'],
        'reference'      : ex['reference'],
        'candidate'      : ex['candidate'],
        'discern_score'  : d['score'],
        'discern_evaluation' : d['evaluation'],
        'discern_error'  : d['error'],
        'mini_discern_score' : m['score'],
        'mini_discern_evaluation' : [e.model_dump() for e in m['entities']] if m['entities'] else None,
        'mini_discern_error' : m['error'],
    })

out_path = out_dir / 'synthetic_discern_results.json'
out_path.write_text(json.dumps(payload, indent=2, default=str))
print(f'Saved → {out_path}')